# Layer Importance Analysis Through Removal Testing

This notebook analyzes the importance of different transformer layers through systematic removal testing.

## Purpose
- Evaluate layer importance through removal testing
- Analyze layer-wise cosine similarities
- Fit and analyze logistic curves to removal patterns
- Identify critical layers in the model architecture

## 1. Setup and Imports

In [ ]:
import os
import sys
import json

# Add project root to path
project_root = os.path.abspath("..")
sys.path.append(project_root)

from src.model_utils import load_model_and_tokenizer
from src.remove_test import (
    compute_layer_cos_sims,
    run_layerwise_remove_test,
    plot_remove_test,
    save_layerwise_results,
    load_layerwise_results,
    fit_logistic_curves,
    plot_fitted_curves,
)

## 2. Configuration

In [ ]:
# Model and path configuration
MODEL_NAME = "Llama-2-13b-hf"
MODEL_ROOT = "/mnt/public/model/huggingface/"
MODEL_PATH = os.path.join(MODEL_ROOT, MODEL_NAME)
OUTPUT_DIR = os.path.abspath("../layer_importance")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Test configuration
PRUNE_RATIOS = [i / 100 for i in range(10, 101, 10)]  # 10% to 100% in 10% steps
USE_SOFTMASK = True
RESULTS_SAVE_PATH = "remove_test_results.npy"

# Test prompts covering different domains
TEXT_LIST = [
    # Mathematics
    "The Pythagorean theorem states that the square of the hypotenuse is equal to the sum of the squares of the other two sides.",
    # Programming
    "In Python, you can define a function using the def keyword followed by the function name and parentheses.",
    # Physics
    "Water boils at 100 degrees Celsius under standard atmospheric pressure.",
    # Astronomy
    "The Earth revolves around the Sun in approximately 365.25 days.",
    # Biology
    "The mitochondrion is often referred to as the powerhouse of the cell because it generates most of the cell's energy."
]

## 3. Model Loading

In [ ]:
print("📥 Loading model and tokenizer...")
model, tokenizer = load_model_and_tokenizer(MODEL_PATH)
tokenizer.pad_token = tokenizer.pad_token or tokenizer.eos_token

## 4. Layer-wise Similarity Analysis

In [ ]:
# Calculate layer-wise cosine similarities
print("📊 Computing layer-wise similarities...")
layer_cos_sims = compute_layer_cos_sims(model, tokenizer, TEXT_LIST)

print("\n=== Layer-wise Input/Output Cosine Similarities ===")
for i, sim in enumerate(layer_cos_sims):
    print(f"Layer {i:02d}: {sim:.4f}")

## 5. Removal Testing

In [ ]:
print("🧪 Running layer-wise removal tests...")
remove_results = run_layerwise_remove_test(
    model,
    tokenizer,
    text_list=TEXT_LIST,
    prune_ratios=PRUNE_RATIOS,
    use_softmask=USE_SOFTMASK
)

# Visualize results for selected pruning ratios
print("\n📈 Plotting removal test results...")
plot_remove_test(remove_results, ratios_to_plot=[0.2, 0.5, 0.8, 1.0])

## 6. Curve Fitting Analysis

In [ ]:
print("📊 Fitting logistic curves to results...")
fitted_result = fit_logistic_curves(
    remove_results=remove_results,
    protect_head_layers=2,
    protect_tail_layers=2,
    outlier_method="iqr",
    outlier_threshold=1.5
)

# Plot fitted curves
plot_fitted_curves(
    fitted_result,
    show_data=True,
    show_logistic=True,
    show_linear=True,
    title="Cosine Similarity Fit (Avg over Ratios)"
)

## 7. Save and Load Results

In [ ]:
# Save results
print("💾 Saving analysis results...")
save_layerwise_results(
    model_name=MODEL_NAME,
    cos_sims=layer_cos_sims,
    remove_results=remove_results,
    fitted_result=fitted_result
)

# Load results (if needed)
print("\n📥 Loading saved results...")
model_name, cos_sims, remove_results, fitted_result = load_layerwise_results(MODEL_NAME)
print("✅ Loaded results for model:", model_name)